# SPECTRE-Plex preprocessing: cycle alignment + bleach subtraction

**Notebook version of `scripts/01_preprocess_spectreplex.py`.**
Run cell-by-cell to inspect intermediate results — useful for choosing parameters and debugging a tricky cycle before committing to a batch run.

## What this does

For multiplex IF acquired in stained/bleach cycles, this notebook:

1. **Best-focus picks** the sharpest z-plane per (cycle, tile, channel) using Brenner gradient (skip=17 between pixels).
2. **Flat-field corrects** with BaSiC — removes per-tile illumination unevenness.
3. **Registers** each bleach image to its matching stained image with PyStackReg (rigid).
4. **Subtracts** the registered bleach (autofluorescence background) from the stained image.

Output is a folder of background-subtracted, in-focus, flat-corrected tiles ready for **Ashlar/McMicro stitching**, then **Cellpose-SAM segmentation**.

## Input expected

A directory layout where each tile/channel/cycle is its own multi-page TIFF z-stack. The bleach round is a separate TIFF acquired *after* photobleaching the stain.

```
data/
├── cycle01/
│   ├── tile_0001_ch00_stained.tif    ← z-stack
│   ├── tile_0001_ch00_bleach.tif     ← z-stack (after bleach)
│   ├── tile_0001_ch01_stained.tif
│   └── ...
├── cycle02/
│   └── ...
```

If your layout differs, edit the file-discovery cell (Section 2). Everything else stays the same.

## Output

```
data_preprocessed/
├── cycle01/
│   ├── tile_0001_ch00_subtracted.tif   ← stained - registered_bleach, flat-corrected, in-focus
│   ├── tile_0001_ch01_subtracted.tif
│   └── ...
```

Feed this folder into Ashlar (see Section 7).


## 1. Install + import

First run only:

```
pip install numpy scipy scikit-image tifffile pyyaml basicpy pystackreg matplotlib tqdm
```

`basicpy` (BaSiC flat-field) needs `jax` — it'll be pulled in automatically.

In [ ]:
# Run once. Skip if already installed.
# %pip install --quiet numpy scipy scikit-image tifffile pyyaml basicpy pystackreg matplotlib tqdm

In [ ]:
from pathlib import Path
import re, time
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import sobel
from skimage import io as skio
from tqdm.auto import tqdm

# Heavy deps — imported lazily so the early cells still run if you don't have them yet
def _basic():
    from basicpy import BaSiC
    return BaSiC
def _stackreg():
    from pystackreg import StackReg
    return StackReg
print("imports ok")

## 2. Paths and channels

Edit these to point at your data. The `RAW_DIR` should contain per-cycle subfolders with stained + bleach z-stacks. `OUT_DIR` will be created.

In [ ]:
RAW_DIR  = Path("data")                         # input  (per-cycle subfolders)
OUT_DIR  = Path("data_preprocessed")             # output (will be created)
OUT_DIR.mkdir(exist_ok=True)

# How to identify stained vs bleach files. Adjust the regex to match your file naming.
STAINED_TAG = "_stained"
BLEACH_TAG  = "_bleach"

# Filename pattern used to extract (tile, channel) so we can pair stained/bleach.
# Default matches  tile_0001_ch00_stained.tif
PAIR_REGEX = re.compile(r"(tile_\d+_ch\d+)")

# Brenner focus skip distance (paper default = 17 px)
BRENNER_SKIP = 17

# List cycles found
cycles = sorted(p for p in RAW_DIR.iterdir() if p.is_dir())
print(f"{len(cycles)} cycle(s) found:")
for c in cycles:
    n_stained = len(list(c.glob(f'*{STAINED_TAG}*.tif*')))
    n_bleach  = len(list(c.glob(f'*{BLEACH_TAG}*.tif*')))
    print(f"  {c.name}:  {n_stained} stained, {n_bleach} bleach")

## 3. Brenner best-focus selection

For each z-stack, pick the plane with the highest **Brenner gradient** — a fast, robust sharpness metric. `skip=17` measures pixel-to-pixel differences across a 17-pixel offset, which is more discriminative for fluorescence images than the standard skip=2.

In [ ]:
def brenner_score(img, skip=BRENNER_SKIP):
    """Brenner gradient sharpness. Higher = sharper."""
    img = img.astype(np.float32)
    dy = img[skip:, :] - img[:-skip, :]
    dx = img[:, skip:] - img[:, :-skip]
    return (dy ** 2).sum() + (dx ** 2).sum()

def best_focus_plane(zstack, skip=BRENNER_SKIP):
    """Return the (best 2D plane, its index, scores array) for a (Z, Y, X) stack."""
    if zstack.ndim == 2:
        return zstack, 0, np.array([brenner_score(zstack, skip)])
    scores = np.array([brenner_score(z, skip) for z in zstack])
    best = int(np.argmax(scores))
    return zstack[best], best, scores

# Demo on one stack
demo_paths = sorted(cycles[0].glob(f'*{STAINED_TAG}*.tif*')) if cycles else []
if demo_paths:
    stack = tifffile.imread(demo_paths[0])
    best, idx, scores = best_focus_plane(stack)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(scores, marker='o')
    ax[0].axvline(idx, color='red', linestyle='--', label=f'best = z{idx}')
    ax[0].set_xlabel('z-plane'); ax[0].set_ylabel('Brenner score')
    ax[0].set_title(demo_paths[0].name); ax[0].legend()
    ax[1].imshow(best, cmap='gray')
    ax[1].set_title(f'best-focus plane (z={idx})'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    print("(no stained TIFFs found yet — set RAW_DIR above)")

## 4. BaSiC flat-field correction

[BaSiC](https://github.com/peng-lab/BaSiCPy) estimates the per-pixel illumination bias (flat-field) + dark-offset from a batch of tiles, then divides it out. Run per-channel: collect all best-focus planes of one channel from one cycle, fit BaSiC, apply.

In [ ]:
def fit_apply_basic(planes_2d_list):
    """Fit BaSiC on a list of 2D planes (same channel), return corrected list."""
    BaSiC = _basic()
    stack = np.stack(planes_2d_list).astype(np.float32)        # (N, Y, X)
    basic = BaSiC(get_darkfield=True, smoothness_flatfield=1.0)
    basic.fit(stack)
    corrected = basic.transform(stack)
    return corrected, basic.flatfield, basic.darkfield

# Demo (one channel, one cycle) — comment out if you have no data yet
if demo_paths:
    # Group demo paths by channel using the regex
    by_ch = {}
    for p in demo_paths:
        m = re.search(r'_ch(\d+)_', p.name)
        if m: by_ch.setdefault(m.group(1), []).append(p)
    if by_ch:
        ch, plist = next(iter(by_ch.items()))
        print(f"fitting BaSiC on channel {ch}, {len(plist)} tiles ...")
        focused = [best_focus_plane(tifffile.imread(p))[0] for p in plist]
        corrected, ff, df = fit_apply_basic(focused)
        fig, ax = plt.subplots(1, 3, figsize=(15, 4))
        ax[0].imshow(focused[0], cmap='gray'); ax[0].set_title('raw');         ax[0].axis('off')
        ax[1].imshow(ff,        cmap='magma'); ax[1].set_title('flat-field');  ax[1].axis('off')
        ax[2].imshow(corrected[0], cmap='gray'); ax[2].set_title('corrected'); ax[2].axis('off')
        plt.tight_layout(); plt.show()

## 5. PyStackReg registration + bleach subtraction

The bleach image is shot *after* photobleaching the stain — same field of view, but stage drift between acquisitions means the bleach and stained images are slightly misaligned. PyStackReg's **rigid** transform (translation + rotation) handles this without warping the cells.

After registration, subtract: `subtracted = stained − registered_bleach`. Clip negatives to zero.

In [ ]:
def register_and_subtract(stained_2d, bleach_2d):
    """Rigid-register bleach to stained, then subtract."""
    StackReg = _stackreg()
    sr = StackReg(StackReg.RIGID_BODY)
    bleach_reg = sr.register_transform(stained_2d.astype(np.float32),
                                       bleach_2d.astype(np.float32))
    sub = stained_2d.astype(np.float32) - bleach_reg
    return np.clip(sub, 0, None), bleach_reg

# Demo on one pair
if cycles:
    stained_files = sorted(cycles[0].glob(f'*{STAINED_TAG}*.tif*'))
    if stained_files:
        s_path = stained_files[0]
        b_path = s_path.with_name(s_path.name.replace(STAINED_TAG, BLEACH_TAG))
        if b_path.exists():
            s = best_focus_plane(tifffile.imread(s_path))[0]
            b = best_focus_plane(tifffile.imread(b_path))[0]
            sub, b_reg = register_and_subtract(s, b)
            fig, ax = plt.subplots(1, 4, figsize=(20, 4))
            for a, im, t in zip(ax, [s, b, b_reg, sub],
                                ['stained', 'bleach (raw)', 'bleach (registered)', 'subtracted']):
                a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
            plt.tight_layout(); plt.show()
        else:
            print(f"no matching bleach for {s_path.name}")

## 6. Batch all cycles

Walks every cycle, pairs stained↔bleach by `(tile, channel)`, picks best focus on each, fits BaSiC per channel, applies, registers, subtracts, and writes a TIFF per stained input.

This is the slow step — it'll typically take a few minutes per cycle on a modern Mac.

In [ ]:
def process_cycle(cycle_dir: Path, out_root: Path):
    """Process one cycle. Returns number of (tile, channel) pairs written."""
    stained_files = sorted(cycle_dir.glob(f'*{STAINED_TAG}*.tif*'))
    if not stained_files:
        print(f"  [skip] no stained files in {cycle_dir.name}")
        return 0
    out_dir = out_root / cycle_dir.name; out_dir.mkdir(parents=True, exist_ok=True)

    # 1) best-focus on every stained AND its bleach pair
    pairs = []   # list of (tile_ch_key, stained_2d, bleach_2d, out_path)
    for s_path in tqdm(stained_files, desc=f"{cycle_dir.name} focus", leave=False):
        m = PAIR_REGEX.search(s_path.name)
        if not m: continue
        key = m.group(1)
        b_path = s_path.with_name(s_path.name.replace(STAINED_TAG, BLEACH_TAG))
        if not b_path.exists():
            print(f"  [warn] no bleach for {s_path.name}; skipping")
            continue
        s_best = best_focus_plane(tifffile.imread(s_path))[0]
        b_best = best_focus_plane(tifffile.imread(b_path))[0]
        out_path = out_dir / s_path.name.replace(STAINED_TAG, '_subtracted')
        pairs.append((key, s_best, b_best, out_path))

    # 2) BaSiC per channel on the stained planes
    by_ch = {}
    for i, (key, s, b, op) in enumerate(pairs):
        m = re.search(r'_ch(\d+)', key)
        ch = m.group(1) if m else 'all'
        by_ch.setdefault(ch, []).append(i)

    s_corrected = [None] * len(pairs)
    b_corrected = [None] * len(pairs)
    for ch, idxs in by_ch.items():
        print(f"  BaSiC channel {ch}: {len(idxs)} tiles")
        s_stack = np.stack([pairs[i][1] for i in idxs])
        b_stack = np.stack([pairs[i][2] for i in idxs])
        s_c, _, _ = fit_apply_basic(list(s_stack))
        # Fit BaSiC separately on bleach because illumination differs slightly post-bleach
        b_c, _, _ = fit_apply_basic(list(b_stack))
        for j, i in enumerate(idxs):
            s_corrected[i] = s_c[j]; b_corrected[i] = b_c[j]

    # 3) register + subtract + write
    n = 0
    for (key, _, _, op), s, b in tqdm(list(zip(pairs, s_corrected, b_corrected)),
                                       desc=f"{cycle_dir.name} subtract", leave=False):
        sub, _ = register_and_subtract(s, b)
        tifffile.imwrite(str(op), sub.astype(np.float32))
        n += 1
    return n

# RUN IT
totals = []
for c in cycles:
    print(f"\n=== {c.name} ===")
    n = process_cycle(c, OUT_DIR)
    totals.append((c.name, n))
print("\nDone:")
for name, n in totals:
    print(f"  {name}: {n} subtracted tiles")

## 7. What's next: stitch → segment → analyze

The preprocessed tiles in `data_preprocessed/` feed into the rest of the SPECTRE-Plex pipeline:

```
data_preprocessed/      ← you are here
   │
   ▼
Ashlar / McMicro        ← external (Nextflow); produces stitched.ome.tif
   │
   ▼
scripts/02_segment_and_type.py        (or cellpose-sam-automation)
   │
   ▼
R/03_spatial_analysis.R               (or demo/run_analysis_and_figures.py)
```

A typical Ashlar invocation:

```
ashlar 'data_preprocessed/cycle*/tile_*_ch00_subtracted.tif' \
       --output stitched.ome.tif \
       --pyramid
```

## Troubleshooting

| Symptom | Fix |
|---|---|
| BaSiC fails with NaN | One channel has all-zero tiles. Skip empty channels or lower `smoothness_flatfield`. |
| PyStackReg rotates wildly | Stained/bleach acquisition gap too large. Try `StackReg.TRANSLATION` instead of `RIGID_BODY`. |
| Subtracted image is mostly zero | Bleach > stained (rare). Inspect with the demo cell in Section 5. |
| Different file naming | Edit `STAINED_TAG`, `BLEACH_TAG`, `PAIR_REGEX` in Section 2. |
| Out of memory on BaSiC | Process fewer tiles per channel at a time; or downsample to 16-bit. |

## Citation

> Anderson MD et al. *SPECTRE-Plex.* Communications Biology 8:636 (2025).
> Peng T et al. *BaSiC.* Nature Communications 8:14836 (2017).
> Thévenaz P et al. *PyStackReg / TurboReg.* IEEE T-IP 7:27 (1998).
